# Chapter 3: Training GMS Stores and Calibrating Thresholds

**Monograph:** *How to Build Policy-Governed AI Agents*

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain why raw geometric scores need calibrated thresholds.
2. Train a GMS store to a depth where calibration is meaningful.
3. Invoke `GMSJudge.calibrate()` to produce a dictionary of calibrated thresholds (`tau_contra`, `geodesic`, `holonomy`, `path_transport`, `phase`).
4. Recognise which threshold each downstream gate consumes.

## Prerequisites

- Chapter 2 notebook completed (familiarity with the six GMS primitives).
- The domain knowledge document `governed_agent/knowledge/data_science_practices.md`.
- CUDA recommended; training takes ~1 minute on GPU.

---
## Preamble

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, CUDA: {torch.cuda.is_available()}")


---
## Lab 3.1 -- Train and Calibrate the Domain Knowledge Store

Chapter 2 trained the `data_science_practices.md` store at 500 epochs to illustrate the primitives. This chapter retrains the same document into a dedicated calibration store. Calibration thresholds are derived from the store's score distribution, so they are tied to the specific trained artifact we calibrate against.

### 3.1.1 Retraining at 500 Epochs

In [ ]:
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.store import GMSExpertStore
from knowlytix.knowledge.ingest import ingest_document

MD_PATH = "governed_agent/knowledge/data_science_practices.md"

config = DocGMSConfig(
    store_path="gms_agent_builder_ext/lab_stores/ch03_domain_500ep",
)
config.train.epochs = 500

store = GMSExpertStore(config, device=DEVICE)
result = ingest_document(
    store, MD_PATH, llm=None, config=config, device=DEVICE,
)

stats = store.stats()
print("\n=== Knowledge Store Statistics ===")
print(f"  Entities:    {stats['entities']}")
print(f"  Relations:   {stats['relations']}")
print(f"  Triples:     {stats['triples']}")
print(f"  ENM entries: {stats['enm_entries']}")

### 3.1.2 Verify True-vs-Fabricated Separation

Before we calibrate, we confirm that 500 epochs produced a separation between true and fabricated triples wide enough for a logistic regression to find a useful cut.

In [3]:
import random
import statistics

random.seed(42)
all_triples = list(store.doc_graph.triples)
entity_names = list(store.adapter.entity_to_idx.keys())

true_triples = [
    (h, r, t) for h, r, t in all_triples
    if store.entity_exists(h) and store.entity_exists(t)
    and store.relation_exists(r)
][:50]

# Fabricate by swapping the tail with a random different entity
fab_triples = []
for h, r, t in true_triples:
    wrong_t = random.choice([e for e in entity_names if e != t])
    fab_triples.append((h, r, wrong_t))

d_true = [store.score_triple(h, r, t)
          for h, r, t in true_triples]
d_true = [x for x in d_true if x is not None]
d_fab = [store.score_triple(h, r, t)
         for h, r, t in fab_triples]
d_fab = [x for x in d_fab if x is not None]

print(f"True triples:       mean={statistics.mean(d_true):.4f}  "
      f"std={statistics.stdev(d_true):.4f}  (n={len(d_true)})")
print(f"Fabricated triples: mean={statistics.mean(d_fab):.4f}  "
      f"std={statistics.stdev(d_fab):.4f}  (n={len(d_fab)})")
print(f"Separation:         "
      f"{statistics.mean(d_fab) - statistics.mean(d_true):+.4f}")

True triples:       mean=1.1803  std=0.1002  (n=50)
Fabricated triples: mean=1.5319  std=0.2000  (n=50)
Separation:         +0.3516


### 3.1.3 Diagnostic Sanity Check

Before calibration we run the nine-check diagnostic suite documented in Appendix F. Three structural invariants (S1--S3) and six primitive checks (P1--P6) probe the geometric substrate and the primitives that downstream gates rely on. Calibrating a store that fails these checks produces thresholds that paper over broken geometry.

The rule: *do not calibrate a store that fails diagnostics*. Fix training first.

In [ ]:
from knowlytix.core.diagnostics import GMSDiagnostics

diag = GMSDiagnostics(
    model=store.model,
    adapter=store.adapter,
    enm=store.enm,
    doc_graph=store.doc_graph,
    device=DEVICE,
)
report = diag.run_all()
GMSDiagnostics.print_report(report)

print(f"\nOverall: "
      f"{'ALL PASSED' if report.all_passed else 'SOME FAILED'}")
print(f"Passed: {report.pass_count}/{report.total}")

### 3.1.4 Running `calibrate()`

`GMSJudge.calibrate()` fits five per-measure logistic regressions against positive and corrupted-tail negative samples drawn from the store's training triples. The derived thresholds are written to `judge._thresholds`, a dictionary that downstream gates read.

In [ ]:
from knowlytix.harness.testing import GMSJudge

judge = GMSJudge(store=store)
calibrations = judge.calibrate(seed=42)

print("\nThresholds stored in judge._thresholds:")
for name, value in judge._thresholds.items():
    if isinstance(value, tuple):
        pretty = ", ".join(f"{v:.4f}" for v in value)
        print(f"  {name:16s} = ({pretty})")
    elif isinstance(value, (int, float)):
        print(f"  {name:16s} = {value:.4f}")
    elif isinstance(value, str):
        print(f"  {name:16s} = {value!r}")
    else:
        print(f"  {name:16s} = {value}")

### 3.1.5 How the Thresholds Are Used Downstream

The thresholds are not applied in this chapter -- they are prepared for other chapters to consume. The consumption pattern is uniform: a downstream module instantiates `PolicyEngine` or `GovernedToolGateway` and passes the `judge` object. The gate's `__init__` reads `judge._thresholds` and overrides its class-level defaults.

The code block below is illustrative; it is not executed here because `GovernedToolGateway` and `RuntimeLearner` with the `tau_contra` override are covered in later chapters.

In [6]:
# Illustrative -- downstream consumption pattern
def _fmt(v):
    if v is None:
        return "(not calibrated)"
    if isinstance(v, tuple):
        return "(" + ", ".join(f"{x:.4f}" for x in v) + ")"
    return f"{v:.4f}"

tau_contra = judge._thresholds.get("tau_contra")
theta_plaus = judge._thresholds.get("geodesic")
tau_holonomy = judge._thresholds.get("holonomy")

print(f"tau_contra     (Ch 6 contradiction gate):   {_fmt(tau_contra)}")
print(f"geodesic       (Ch 7 plausibility check):   {_fmt(theta_plaus)}")
print(f"holonomy       (Ch 6 reasoning consistency): {_fmt(tau_holonomy)}")

# Downstream gates read the same dictionary, e.g.:
#
#   from knowlytix.harness.governance.governed_gateway import GovernedToolGateway
#   gw = GovernedToolGateway(
#       session=session,
#       store=store,
#       theta_plausibility=judge._thresholds["geodesic"],
#       tau_contra=judge._thresholds["tau_contra"],
#       ...
#   )
#
# The gate does not re-calibrate; it inherits the thresholds.


tau_contra     (Ch 6 contradiction gate):   (not calibrated)
geodesic       (Ch 7 plausibility check):   1.3590
holonomy       (Ch 6 reasoning consistency): 2.2398


---
## Exercises

### Exercise 3.1 -- Sensitivity to Training Depth

Train the same knowledge store at 100, 200, 300, 500 and 1000 epochs. For each, run `calibrate()` and record `tau_contra`. Plot the resulting curve. At what epoch does the threshold stabilise? What does a still-moving threshold indicate about the store's readiness?

In [7]:
# === Exercise 3.1: Your code here ===
# Hint: loop over [100, 200, 300, 500, 1000]; use a distinct store_path
# for each epoch count; call judge.calibrate() after each train.
pass

### Exercise 3.2 -- Recalibrate on a Second Corpus

Create `adult_income_domain.md` with 15-20 facts about the Adult Income dataset (protected attributes, fairness metrics, target definition). Ingest it into a new store, run `calibrate()` and compare the resulting thresholds with those of the data-science practices store. Which measures shift the most? Why might `tau_contra` differ between the two corpora?

In [8]:
# === Exercise 3.2: Your code here ===
pass

### Exercise 3.3 -- Manual Threshold vs Calibrated Threshold

Using the 500-epoch store from this chapter, pick 20 pairs of entities. Hand-label each pair as *agree*, *unrelated* or *contradict*. Compute `tension_energy` for each pair. Compare classification accuracy using (a) the calibrated `tau_contra` and (b) the Chapter 2 default of 1.7. Quantify the decision difference.

In [9]:
# === Exercise 3.3: Your code here ===
pass

---
## Summary

This notebook:

1. Retrained the data-science-practices knowledge store at 500 epochs into a dedicated calibration store.
2. Verified the true-vs-fabricated separation had widened relative to the Chapter 2 illustrative run.
3. Ran the nine-check diagnostic suite before calibration.
4. Invoked `GMSJudge.calibrate()` to produce five per-measure thresholds.
5. Previewed how downstream gates (Ch 6 contradiction gate, Ch 7 tool gateway) consume the calibrated dictionary.

The resulting `judge._thresholds` dictionary is the artifact the rest of the monograph builds on. Retraining the store is the trigger for recalibration; downstream gates inherit, they do not re-calibrate.